# Module 1: Build the Graph

**Overview**

This module turns five hotel documents into a graph that later modules can search.

- **Text layer:** `Document` and `Chunk` nodes store the source text. Each `Chunk` stores text and a 1024-dimension embedding. Vector and keyword searches use this layer.
- **Fact layer:** `Hotel`, `Room`, `Amenity`, `Policy`, and `Service` nodes store facts from the text. Cypher queries use this layer.
- **Graph connections:** `FROM_DOCUMENT` and `FROM_CHUNK` link the text to its facts. A search can find a chunk, its source document, and the connected hotel facts.

An embedding finds text with a similar meaning. It does not store exact facts such as a hotel's city, whether it has a spa, or whether the spa costs extra. This module writes those facts as nodes and relationships so queries can match them exactly.

```text
hotel-tokyo-002.txt
    |  one source file, about 7 KB of text
    v
(:Document {source_filename: "hotel-tokyo-002.txt"})
    ^
    |  FROM_DOCUMENT                      the lexical layer: the text
(:Chunk {text, embedding: 1024 floats})
    ^
    |  FROM_CHUNK                         the domain layer: the facts
(:Hotel {name, address, guest_rating, total_rooms, email, phone})
    |
    +-[:HAS_ROOM]---------> (:Room {type, bed_configuration, max_occupancy, min_rate})
    +-[:OFFERS_AMENITY]---> (:Amenity {name})
    +-[:HAS_POLICY]-------> (:Policy {name, description})
    +-[:PROVIDES_SERVICE]-> (:Service {name, description, cost, hours})
```

Every domain relationship starts at `Hotel`. The provenance relationships connect documents and chunks to the extracted facts. Each document creates a one-hop set of connected facts. Every later module reads both layers.

## Extract facts and create the graph

Claude on Amazon Bedrock extracts facts from prose. `SimpleKGPipeline` from `neo4j-graphrag` runs the extraction with a fixed schema. The source already lists amenities as a bullet list, so a parser reads that list directly. It creates one shared `Amenity` node for each exact label.

Use the LLM for prose. Parse a structured list directly when the source already contains one. After it writes the graph, the build creates two retrieval indexes and two lookup indexes.

## Add five hotels to the preloaded graph

The dump restored during Setup contains the preloaded corpus. It uses the same LLM and parser boundary. Building the full corpus takes hours, so this notebook reserves five documents for you to build. The build takes about four minutes. It adds five hotel graphs that later modules can query.

## Set up Python imports

`start_module` puts two directories on the Python import path, then reads the workshop's configuration files.

- **`notebooks/`:** This directory contains the shared `workshop` package.
- **`02-connected-context`:** This directory contains `graph_builder.py`, the extraction code used here.

Module 1 and Module 2 use the same extraction code, so it lives in one shared directory.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module(
    "01-build-graph", extra_import_dirs=("02-connected-context",)
)
print(f"Workshop root: {REPO_ROOT}")

In [ ]:
# Bedrock needs a region, and botocore reads only AWS_DEFAULT_REGION, never
# AWS_REGION. This sets both from one resolved value so that clients built
# without an explicit region_name land in the workshop's region instead of
# whatever the active AWS profile happens to configure.
from workshop.aws_region import configure_aws_region

print(f"Region: {configure_aws_region()}")

## Check the graph before the build

The restored dump contains the preloaded corpus documents. It uses the same LLM and parser boundary as this module. The dump does not include the vector and full-text indexes required for retrieval. It also does not include the five documents that you will extract.

Record the current document and hotel counts. You will compare them with the counts after the build. Expect one `Hotel` for each `Document`. The build stops if a source produces zero or multiple complete `Hotel` nodes, or if two sources share one `Hotel` node.

In [ ]:
from graph_builder import connect, count_documents
from workshop.graph_connection import graph_database, require_neo4j_env

require_neo4j_env()

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        hotels_before = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]
    documents_before = count_documents(driver)

print(f"Database: {graph_database()}")
print(f"Documents already loaded: {documents_before}")
print(f"Hotels already loaded:    {hotels_before}")

### Inspect a preloaded hotel's text and facts

The next cell reads one hotel from the restored dump. It reports the number of chunks, the embedding width, and the connected rooms, amenities, policies, and services. The last cell runs the same check for the first hotel you extract. Use this result as the baseline. This cell only reads data.

In [ ]:
# Read-only. Reports the lexical and domain layers for one document.
from graph_layers import show_both_layers

show_both_layers("hotel-paris-001.txt")

## Load the five documents for this build

This build extracts the `-002` document for Tokyo, Sydney, Rio de Janeiro, Cape Town, and Prague. These files keep the build separate from later-module fixtures.

- **Later-module fixtures:** No later module asks about these five `-002` hotels, so this build cannot overwrite a fixture.
- **City coverage:** Each city still has its `-001` hotel in the dump while you extract the `-002` hotel.
- **Cairo retrieval test:** Cairo is excluded. Module 2 starts its retrieval comparison with a Cairo hotel, and that comparison must come from the preloaded graph.

The next cell unpacks the five files from the corpus archive and previews the first one. Read the preview. It gives the hotel's name, address, rating, and contact details in prose under headings. It lists rooms, amenities, policies, and services later in the file. The extraction turns this content into typed graph nodes.

In [ ]:
from held_out_documents import HELD_OUT_DOCUMENTS, extract_held_out

paths = extract_held_out()
for path in paths:
    print(f"  {path.name}  ({path.stat().st_size:,} bytes)")

print(f"\n--- {paths[0].name}, first 400 characters ---")
print(paths[0].read_text(encoding="utf-8")[:400])

## Extract facts and write graph nodes

`SimpleKGPipeline` runs five stages for each document. A deterministic amenity step runs after the pipeline.

| Stage | Component | Purpose in this build |
|-------|-----------|-----------------------|
| Split | `FixedSizeSplitter` | Cuts the document into text slices of at most `CHUNK_SIZE` characters |
| Embed | Amazon Nova, from Setup's model table | Turns each text slice into a 1024-dimension vector and stores it on the `Chunk` node |
| Extract | Claude on Amazon Bedrock | Reads the `Chunk` text and returns JSON restricted to the LLM schema |
| Resolve | `perform_entity_resolution=False` | Keeps same-name hotels in different cities as distinct nodes |
| Write | The pipeline's Neo4j writer | Creates the `Document`, `Chunk`, and entity nodes, then connects them |
| Amenities | Deterministic parser | Reads the authored bullets and merges shared nodes by exact amenity name |

**Chunk size:** This setting controls how much text the model reads in one call. The largest corpus document is 7,442 bytes. `CHUNK_SIZE` is 12000, so each document becomes exactly one `Chunk` node. The model receives the hotel's name, address, rating, rooms, policies, and services in the same call. `CHUNK_OVERLAP` is 0 because each document has one chunk. The build compares total `Chunk` and document counts at the end. More `Chunk` nodes than documents means a document was split and processed in two calls.

**Response length:** One hotel's JSON can exceed the 4096-token default set by the workshop's Bedrock client. A response that reaches this limit ends in the middle of the JSON and fails. The document fails visibly and receives one retry. `EXTRACTION_MAX_TOKENS` is 16000 so the full response fits.

These three settings are in `graph_config.py`, beside the build code.

**Amenity identity:** After LLM extraction succeeds, the build requires one `Hotel` for each source document. It reads only the bullets under `## Hotel Amenities` and stores each exact label as `Amenity.name`. Matching labels use one shared node. Global name-based resolution stays off so hotels with the same display name remain separate.

## Use a fixed graph schema

`SimpleKGPipeline` can extract without a schema. Without a schema, the model chooses labels from each document's headings. Different documents use different headings, so the graph structure changes between documents.

| Kind of drift | Labels the model chose | Why it breaks queries |
|---------------|------------------------|-----------------------|
| A property promoted to a node | `Address`, `Fee`, `Location` | The address sits on the hotel node in one document and one hop away in the next |
| A type split from its instance | `RoomType`, `BedConfiguration` | A room's own properties become separate nodes to join through |
| Two names for one thing | `ContactMethod`, `ContactInfo` | Both are reasonable, and a query has to know which one a given document used |
| Geography expanded into a hierarchy | `City`, `Country` | The city is text inside the address in most documents and a node in a few |

A Cypher pattern cannot match every structure in this table. The build needs one vocabulary for every document. The schema defines that vocabulary before extraction and applies it to every document.

The LLM schema controls prose extraction. It defines four node types, three relationship types, and the allowed patterns. It sets `additional_node_types`, `additional_relationship_types`, and `additional_patterns` to `False`. The model must drop a fact when it has no allowed label.

The schema's property descriptions give the model direct instructions:

- **`address`:** "Never model the address as its own node" keeps `Address` out of the graph.
- **`guest_rating`:** Read `4.6` from `4.6/5.0` and store it as a float. Later modules can average numeric values.

Module 2 compares source retrieval with graph-enriched retrieval. The graph result returns `name`, `address`, and `guest_rating` from the `Hotel` node. The fixed schema keeps that result consistent.

### Read the amenity list with code

The source lists amenities as bullets under `## Hotel Amenities`. The build reads this structured list directly so each authored value stays exact.

1. The LLM schema excludes `Amenity` and `OFFERS_AMENITY`.
2. Code reads the amenity bullets and stops at the next heading.
3. The exact trimmed bullet text becomes the shared `Amenity.name`.
4. The relationship keeps source provenance so the build can compare graph facts with the file.

A sentence such as "Pool facilities are not available at this property" appears outside the list. It cannot create a positive Pool amenity.

Use the LLM for prose. Parse a structured list directly when the source already contains one. The prebuilt graph and these five documents use the same rule.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA, LLM_EXTRACTION_SCHEMA, OFF_SCHEMA_LABELS

print("Node types the LLM extraction is allowed to produce:")
for node_type in LLM_EXTRACTION_SCHEMA["node_types"]:
    properties = ", ".join(p["name"] for p in node_type.get("properties", []))
    print(f"  :{node_type['label']:<9} {properties}")

print("\nRelationships it is allowed to produce:")
for start, rel, end in LLM_EXTRACTION_SCHEMA["patterns"]:
    print(f"  (:{start})-[:{rel}]->(:{end})")

amenity = next(
    node for node in GRAPH_SCHEMA["node_types"] if node["label"] == "Amenity"
)
amenity_properties = ", ".join(p["name"] for p in amenity["properties"])
print("\nAdded deterministically after LLM extraction:")
print(f"  :Amenity {amenity_properties}")
print("  (:Hotel)-[:OFFERS_AMENITY]->(:Amenity)")

print("\nLabels an unpinned run invents instead, which the build treats as a failure:")
print(f"  {', '.join(OFF_SCHEMA_LABELS)}")

### Optional: compare labels without the schema

Set `RUN_UNPINNED_DEMO` to `True` to extract **one** document without a schema. The cell prints the labels created by the LLM. Compare them with the four kinds of drift above. This comparison uses model tokens. The main build runs whether you run this comparison or skip it.

The demo uses a source filename reserved for this temporary comparison. It removes its `Document`, `Chunk`, and invented label nodes after extraction succeeds or fails. It does not change the five participant documents or the preloaded graph.

In [ ]:
# Optional. Extracts one document with no schema and reports what it invented.
RUN_UNPINNED_DEMO = False
UNPINNED_DEMO_SOURCE_FILENAME = "demo-unpinned-schema-comparison.txt"

if RUN_UNPINNED_DEMO:
    from graph_builder import clear_document, session as build_session, snapshot_chunk_ids
    from neo4j_graphrag.components.text_splitters.fixed_size_splitter import (
        FixedSizeSplitter,
    )
    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

    from graph_config import CHUNK_OVERLAP, CHUNK_SIZE
    from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM

    sample = paths[0]
    driver = connect()
    try:
        baseline = snapshot_chunk_ids(driver)
        unpinned = SimpleKGPipeline(
            llm=BedrockLLM(),
            driver=driver,
            embedder=BedrockEmbeddings(),
            schema=None,  # the whole point
            text_splitter=FixedSizeSplitter(
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
            ),
            from_pdf=False,
            perform_entity_resolution=False,
        )
        await unpinned.run_async(
            file_path=UNPINNED_DEMO_SOURCE_FILENAME,
            text=sample.read_text(encoding="utf-8"),
            document_metadata={
                "source_filename": UNPINNED_DEMO_SOURCE_FILENAME,
            },
        )
        new_chunks = list(snapshot_chunk_ids(driver) - baseline)
        with build_session(driver) as neo4j_session:
            invented = neo4j_session.run(
                """
                MATCH (c:Chunk)<-[:FROM_CHUNK]-(n)
                WHERE elementId(c) IN $ids
                UNWIND [l IN labels(n) WHERE NOT l STARTS WITH '__'] AS label
                RETURN label, count(*) AS count ORDER BY count DESC
                """,
                ids=new_chunks,
            ).values()
        print(f"Unpinned extraction produced temporary labels: {invented}")
    finally:
        clear_document(driver, UNPINNED_DEMO_SOURCE_FILENAME)
        driver.close()
else:
    print("Skipped. Set RUN_UNPINNED_DEMO = True to watch the labels drift.")

## Build the graph from the five documents

`run_additive_build` adds the five documents to the existing graph in this order:

1. Clears earlier copies of these five documents only.
2. Parses all five amenity lists before opening the graph. Malformed source structure fails early.
3. Records the document count and current `Chunk` element IDs so the run can identify its own work.
4. Extracts prose facts one document at a time, with a 180-second limit for each document.
5. Retries each failure once. It clears the document before retrying unless a write for that document started in the last 30 seconds.
6. Requires one `Hotel` per source, attaches the authored amenities, and checks the exact source-filename and amenity-name pairs.
7. Checks the schema, creates the two retrieval indexes and two lookup indexes, and checks requirements for later modules.

You can safely rerun this cell. The pipeline uses `CREATE` instead of `MERGE`, so a retry without the clear would leave a second `Document` and `Chunk` for the same file. The clear uses `source_filename` and only the five reserved filenames. It leaves the preloaded documents unchanged.

Bedrock can throttle when many participants extract at once. The AWS client uses adaptive backoff. It slows each client before retrying so they do not all retry against the same regional quota at full speed. Rerun this cell if a document still fails.

The build runs five checks. A failed check stops the build:

- **Schema:** The check lists labels from this run's chunks and fails when it finds an off-schema label.
- **One Hotel per source:** The check fails for zero or multiple hotels, or for one hotel shared by multiple source documents.
- **Amenity source match:** The check compares exact source-filename and amenity-name pairs.
- **Index contract:** The check reads all four indexes and compares their type, state, label, and property. It also checks the vector dimensions and similarity function.
- **Later-module questions:** The check runs queries for requirements such as Paris hotels with ratings and Cairo hotels with a spa, a pool, and a rating.

All five documents must load. Each must create exactly one `Chunk` and one `Hotel`, and every amenity pair must match its source list. The fixture checks allow variation in LLM-extracted properties. They require at least one Cairo hotel with a spa, a pool, and a rating, plus at least two Paris hotels with a rating. One missing optional LLM-extracted property can vary. An off-schema label, a missing `Hotel`, or an amenity mismatch means the graph contract failed and stops the build.

Expect about four minutes. Each document prints a line when it finishes.

In [ ]:
from graph_builder import run_additive_build

exit_code = await run_additive_build(paths, "Module 1: building your five hotels")

if exit_code != 0:
    raise RuntimeError(
        "The build did not finish cleanly. Read the output above, then re-run "
        "this cell; it clears only your five documents before retrying."
    )

## Create the workshop indexes

The build creates all four indexes after it writes the graph. The retrieval indexes cover every `Chunk`, including the chunks from your five documents. The lookup indexes make repeated source-file and hotel-name matches efficient on restored and rebuilt graphs.

| Index | What it reads | What it finds |
|-------|---------------|---------------|
| `hotel_chunk_embeddings` | `Chunk.embedding`, cosine similarity over 1024 dimensions | Text that means the same thing as the question in different words |
| `hotel_chunk_fulltext` | `Chunk.text`, full-text | Exact strings that embeddings blur together, such as a postal code or a hotel name |
| `workshop_document_source_filename` | `Document.source_filename`, range | A source document by its recorded filename |
| `workshop_hotel_name` | `Hotel.name`, range | Hotels matching a stored name |

Each retrieval index serves a different query. The embedding for `60611` is close to other five-digit numbers, so vector search can rank the matching `Chunk` low. Full-text search matches `60611` exactly. A question may use words that the document does not contain. Vector search can still find text with the same meaning. Module 2 compares these signals. Module 3 then uses two tested read paths: passage search and structured record queries.

Use the same model, dimensions, and purpose for document and query embeddings. The model that writes document embeddings must also embed queries at the same width and for the same purpose. Different dimensions can return incorrect rows without an error. The workshop's embedder has no environment override because its model and width are set in code.

Index creation is idempotent, then waits for every index to come online. You can safely repeat this step. Later retrieval waits for the indexes to finish building. The next cell reads all four indexes from the database.

In [ ]:
# Read-only. Reports the four indexes the build just created.
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    DOCUMENT_SOURCE_FILENAME_INDEX,
    HOTEL_NAME_INDEX,
)

INDEX_QUERY = """
SHOW INDEXES YIELD name, type, state, labelsOrTypes, properties, options
WHERE name IN $names
RETURN name, type, state, labelsOrTypes, properties, options
ORDER BY name
"""

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        indexes = session.run(
            INDEX_QUERY,
            names=[
                CHUNK_VECTOR_INDEX,
                CHUNK_FULLTEXT_INDEX,
                DOCUMENT_SOURCE_FILENAME_INDEX,
                HOTEL_NAME_INDEX,
            ],
        ).data()

if not indexes:
    print("No workshop index exists yet. Run the build cell above.")

for index in indexes:
    config = (index["options"] or {}).get("indexConfig", {})
    labels = ", ".join(index["labelsOrTypes"])
    properties = ", ".join(index["properties"])
    print(index["name"])
    print(f"  type:       {index['type']}")
    print(f"  state:      {index['state']}")
    print(f"  indexes:    :{labels}({properties})")
    if index["type"] == "VECTOR":
        print(f"  dimensions: {config.get('vector.dimensions')}")
        print(f"  similarity: {config.get('vector.similarity_function')}")

## Check the hotels you added

The next cell lists the five hotels you extracted with their addresses, ratings, and amenity counts. It also compares document and hotel counts before and after the build.

Check the rating column. These values are floats on `Hotel` nodes, so later queries can average them. Check the amenity counts. A document creates an amenity node only when it affirms that amenity, so the count is exact.

The following cell inspects both layers for the first hotel you extracted. It uses the same check as the preloaded hotel earlier in this notebook.

In [ ]:
with connect() as driver:
    with driver.session(database=graph_database()) as session:
        print("The hotels you just extracted:\n")
        for record in session.run(
            """
            MATCH (d:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(h:Hotel)
            WHERE d.source_filename IN $filenames
            OPTIONAL MATCH (h)-[:OFFERS_AMENITY]->(a:Amenity)
            WHERE a.name IS NOT NULL
            WITH h, count(DISTINCT a) AS amenities
            RETURN h.name AS name, h.address AS address,
                   h.guest_rating AS rating, amenities
            ORDER BY name
            """,
            filenames=list(HELD_OUT_DOCUMENTS),
        ):
            print(f"  {record['name']}")
            print(f"    {record['address']}")
            print(f"    rating {record['rating']}, {record['amenities']} amenities\n")

        hotels_after = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]

    documents_after = count_documents(driver)

print(f"Documents: {documents_before} -> {documents_after}")
print(f"Hotels:    {hotels_before} -> {hotels_after}")
print(
    "\nEvery aggregation and connected traversal from here on runs across the whole "
    "graph, yours included."
)

In [ ]:
# Read-only. The same both-layers walk, now on a hotel you extracted yourself.
show_both_layers(HELD_OUT_DOCUMENTS[0])

### Check one shared amenity

Both Chicago source files include the exact bullet `Complimentary High-Speed Wifi`. The next cell starts at each source document, follows its `Hotel`, and reaches the same `Amenity` node. Two source filenames on one node confirm that the deterministic amenity parser shares exact labels.

In [ ]:
# Read-only. Proves that two source Hotels traverse to one shared Amenity node.
CHICAGO_SOURCES = ["hotel-chicago-001.txt", "hotel-chicago-002.txt"]
CHICAGO_WIFI_QUERY = """
CYPHER 25
MATCH (document:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(hotel:Hotel)
MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity {name: $amenity_name})
WHERE document.source_filename IN $filenames
RETURN elementId(amenity) AS amenity_node_id,
       collect(DISTINCT hotel.name) AS hotels,
       collect(DISTINCT document.source_filename) AS source_filenames
"""

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        shared_wifi = session.run(
            CHICAGO_WIFI_QUERY,
            filenames=CHICAGO_SOURCES,
            amenity_name="Complimentary High-Speed Wifi",
        ).data()

if len(shared_wifi) != 1 or set(shared_wifi[0]["source_filenames"]) != set(CHICAGO_SOURCES):
    raise RuntimeError("The two Chicago Hotels do not share the authored WiFi node.")

print(shared_wifi[0])

## Continue to Module 2

**Purpose:** Use this graph for GraphRAG retrieval.

You now have searchable source chunks, connected hotel facts, and vector and full-text indexes. Module 2 compares GraphRAG read paths over this graph.

In [ ]:
from workshop.workshop_utils import lego_progress

lego_progress(1)